# 03. EDA: синтетический data room (dev) и сравнение с реальными данными

**Зачем.** MVP проверяется на синтетике (P2). Этот разбор отвечает на четыре вопроса:

1. Что лежит в синтетическом data room: объём, состав, качество, траектории метрик.
2. Как устроена эталонная разметка: баланс вердиктов, случаи каталога, сколько утверждений лежит у порогов P5.
3. Какие данные нужны для какой метрики и есть ли они (вопрос ревьюера ДЗ №1).
4. Чем синтетика отличается от реальных данных: биллинг (прокси — UCI Online Retail II, EDA 02) и формулировки
   KPI в проспектах SEC (EDA 01). Это риски R2 и R3 дизайн-документа.

**Что читаем.** Только dev-сплит (`data/synthetic/dev/*`, 5 компаний) и агрегаты
`data/synthetic/holdout_summary.json`. **Каталоги holdout-компаний не открываются** — ни файлы, ни манифесты.

Выводы — в `reports/eda/03_synthetic_dataroom.md`, графики — `reports/figures/03_*.png`.

In [ ]:
import importlib.util
import json
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

from dd_verifier.data import (
    company_dirs,
    drop_exact_duplicates,
    load_claims,
    load_cohorts,
    load_data_room,
    load_documents,
    load_truth,
    read_tables,
    run_quality_checks,
    split_slides,
)
from dd_verifier.metrics import (
    DataRoom,
    NotEnoughData,
    arr,
    compute,
    evaluate_claim,
    month_range,
    tolerance_for,
)
from dd_verifier.metrics.tolerance import metric_meta
from dd_verifier.synth.generator import (
    assign_cases,
    build_company,
    company_configs,
    generate,
    load_catalog,
    load_yaml,
)
from dd_verifier.synth.noise import add_noise

SYN = Path("data/synthetic")
FIG = Path("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)
CONFIG = Path("configs/synth.yaml")
if not (SYN / "dev").exists():
    generate(CONFIG, SYN)  # ~4 с; holdout тоже генерируется, но не читается

DEV = company_dirs(SYN, "dev")  # только dev: holdout-каталоги не открываем
COMPANIES = [d.name for d in DEV]
HOLDOUT = json.loads((SYN / "holdout_summary.json").read_text(encoding="utf-8"))
END = "2025-12"
print(COMPANIES)

# Оформление графиков: палитра и «тихие» оси (reference palette dataviz-skill)
C = ["#2a78d6", "#eb6834", "#6250d6", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#e34948"]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
VERDICTS = [
    "SUPPORTED",
    "EXAGGERATED",
    "CONTRADICTED",
    "NOT_ENOUGH_DATA",
    "UNVERIFIABLE",
    "FACTUAL_OUT_OF_SCOPE",
]
VCOLOR = dict(zip(VERDICTS, C[:6], strict=True))
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK2,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "font.size": 9,
        "legend.frameon": False,
        "lines.linewidth": 2,
    }
)


def save(fig, name: str) -> None:
    path = FIG / f"03_{name}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    print(f"{path}: {path.stat().st_size / 1024:.0f} KB")

## 1. Объём: таблицы и документы по компаниям

Строки считаем по файлам «как пришли» (с шумом). Для holdout — только агрегаты из `holdout_summary.json`.

In [ ]:
TABLES = {d.name: read_tables(d) for d in DEV}
CLAIMS = pd.concat([load_claims(d) for d in DEV], ignore_index=True)
CLAIMS["qualifier"] = CLAIMS["qualifier"].fillna("exact")
CLAIMS["case"] = CLAIMS["case_id"].fillna("plain")
DOCS = {d.name: load_documents(d) for d in DEV}
MANIFEST = {d.name: json.loads((d / "manifest.json").read_text(encoding="utf-8")) for d in DEV}

rows = []
for d in DEV:
    t, truth, docs = TABLES[d.name], load_truth(d), DOCS[d.name]
    cl = CLAIMS[CLAIMS["company_id"] == d.name]
    rows.append(
        {
            "компания": d.name,
            "архетип": MANIFEST[d.name]["params"]["archetype"],
            "история": f"{truth['month'].iloc[0]}..{truth['month'].iloc[-1]}",
            "месяцев": len(truth),
            "customers": len(t["customers"]),
            "invoices": len(t["invoices"]),
            "fx_rates": len(t["fx_rates"]),
            "документов": len(docs),
            "слов в документах": sum(len(x.split()) for x in docs.values()),
            "утверждений": len(cl),
            "семейств": cl["family_id"].nunique(),
        }
    )
VOL = pd.DataFrame(rows).set_index("компания")
num = VOL.select_dtypes("number").columns.drop("месяцев")
VOL.loc["итого dev", num] = VOL[num].sum()
print(VOL.to_string())

hold_rows = pd.DataFrame(HOLDOUT["row_counts"]).T
print("\nholdout (10 компаний, агрегаты):")
print(hold_rows.to_string())
print("утверждений:", HOLDOUT["claims_total"], HOLDOUT["claims_per_company"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
v = VOL.drop("итого dev")
for ax, col, title in zip(
    axes,
    ["invoices", "customers", "утверждений"],
    ["Строк в invoices.csv", "Записей в customers.csv", "Утверждений в gold"],
    strict=True,
):
    ax.barh(v.index, v[col], color=C[0], height=0.6)
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
    for i, x in enumerate(v[col]):
        ax.text(x, i, f" {int(x):,}".replace(",", " "), va="center", color=INK2, fontsize=8)
    ax.set_xlim(0, v[col].max() * 1.25)
axes[0].invert_yaxis()
fig.suptitle("Объём dev-сплита по компаниям", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "volume")

## 2. Состав: сегменты, каналы, валюты, типы платежей, статусы

Доли выручки считаем по леджеру библиотеки метрик (`DataRoom.ledger`: void исключены, кредит-ноты вычтены,
выручка разнесена по месяцам периода услуги, курс — месяца начала услуги) за 2025 год.

In [ ]:
DR = {d.name: load_data_room(d) for d in DEV}

comp = {}
for name in COMPANIES:
    t, dr = TABLES[name], DR[name]
    cust = t["customers"]
    led = dr.ledger[dr.ledger["month"].between("2025-01", END)]
    inv = drop_exact_duplicates(t["invoices"])[0]
    led = led.merge(
        inv.drop_duplicates("invoice_id")[["invoice_id", "currency"]], on="invoice_id", how="left"
    ).merge(cust[["customer_id", "channel", "segment"]], on="customer_id", how="left")
    rev = led["net_usd"].sum()
    live = cust[cust["go_live_date"].notna()]
    comp[name] = {
        "segment": live["segment"].value_counts(normalize=True),
        "channel": led.groupby("channel")["net_usd"].sum() / rev,
        "currency": led.groupby("currency")["net_usd"].sum() / rev,
        "payment_type": led.groupby("payment_type")["net_usd"].sum() / rev,
        "status": t["invoices"]["status"].value_counts(normalize=True),
    }

for dim in ["segment", "channel", "currency", "payment_type", "status"]:
    tab = pd.DataFrame({n: comp[n][dim] for n in COMPANIES}).fillna(0).T * 100
    print(f"\n{dim}, %:")
    print(tab.round(1).to_string())

pilots = {n: int(TABLES[n]["customers"]["is_pilot"].sum()) for n in COMPANIES}
not_live = {
    n: int(
        (TABLES[n]["customers"]["go_live_date"].isna() & ~TABLES[n]["customers"]["is_pilot"]).sum()
    )
    for n in COMPANIES
}
prepay = {
    n: float(
        (
            (
                TABLES[n]["invoices"]["service_period_end"]
                - TABLES[n]["invoices"]["service_period_start"]
            ).dt.days
            > 300
        ).mean()
        * 100
    )
    for n in COMPANIES
}
print("\nпилотов:", pilots)
print("подписаны, но не запущены (не пилоты):", not_live)
print("доля инвойсов с годовым периодом услуги, %:", {k: round(v, 1) for k, v in prepay.items()})

In [ ]:
DIMS = [
    ("segment", "Сегмент (запущенные клиенты)", ["SMB", "MidMarket", "Enterprise"]),
    ("channel", "Канал (доля net-выручки 2025)", ["direct", "reseller"]),
    ("currency", "Валюта (доля net-выручки 2025)", ["USD", "EUR", "GBP"]),
    (
        "payment_type",
        "Тип платежа (доля net-выручки 2025)",
        ["subscription", "implementation", "one_time"],
    ),
]
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9), sharey=True)
for ax, (dim, title, cats) in zip(axes, DIMS, strict=True):
    left = np.zeros(len(COMPANIES))
    for k, cat in enumerate(cats):
        vals = np.array([comp[n][dim].get(cat, 0.0) * 100 for n in COMPANIES])
        ax.barh(
            COMPANIES,
            vals,
            left=left,
            color=C[k],
            height=0.6,
            label=cat,
            edgecolor="#fcfcfb",
            linewidth=1,
        )
        for i, (lft, w) in enumerate(zip(left, vals, strict=True)):
            if w >= 12:
                ax.text(
                    lft + w / 2,
                    i,
                    f"{w:.0f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="#0b0b0b" if C[k] == "#eda100" else "white",
                )
        left += vals
    ax.set_title(title, loc="left", fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_xlabel("%")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=7, handlelength=1)
axes[0].invert_yaxis()
fig.suptitle("Состав data room по компаниям, %", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "composition")

## 3. Траектории MRR/ARR: стандартное определение и определения компании

ARR на каждый месяц считаем библиотекой метрик в четырёх вариантах онтологии: `standard`,
`incl_one_time` (D01: в ARR попали платежи за внедрение и разовые), `gross_channel` (D05: реселлерские
продажи по gross), `peak_month` (D02: аннуализирован пиковый месяц). Точки — заявленные в документах
значения ARR/MRR; для утверждений о росте — ARR, который следует из заявленного множителя
(стандартный ARR на начало × заявленный множитель).

In [ ]:
ARR_VARIANTS = ["standard", "incl_one_time", "gross_channel", "peak_month"]


def arr_series(dr: DataRoom) -> pd.DataFrame:
    months = month_range(*dr.coverage)
    out = {}
    for v in ARR_VARIANTS:
        vals = []
        for m in months:
            try:
                vals.append(arr(dr, m, v).value)
            except NotEnoughData:
                vals.append(np.nan)
        out[v] = vals
    return pd.DataFrame(out, index=months)


ARR = {n: arr_series(DR[n]) for n in COMPANIES}
gap_end = (
    pd.DataFrame({n: ARR[n].loc[END] / ARR[n].loc[END, "standard"] - 1 for n in COMPANIES}).T.drop(
        columns="standard"
    )
    * 100
)
arr_end = pd.DataFrame({n: ARR[n].loc[END] / 1e3 for n in COMPANIES}).T
print("ARR 2025-12, тыс. $:")
print(arr_end.round(0).to_string())
print("\nразрыв варианта к стандарту на 2025-12, %:")
print(gap_end.round(1).to_string())
print("\nпосаженные случаи:", {n: MANIFEST[n]["params"]["cases_planted"] for n in COMPANIES})

In [ ]:
def claimed_arr_points(name: str) -> pd.DataFrame:
    cl = CLAIMS[
        (CLAIMS["company_id"] == name) & CLAIMS["metric_id"].isin(["arr", "mrr", "arr_growth"])
    ]
    cl = cl[cl["claim_type"].isin(["level", "growth"])].drop_duplicates("family_id")
    pts = []
    for r in cl.itertuples():
        if r.metric_id == "arr_growth":
            start = r.period_start
            y = ARR[name].loc[start, "standard"] * float(r.claimed_value)
        elif r.metric_id == "mrr":
            y = float(r.claimed_value) * 12
        else:
            y = float(r.claimed_value)
        pts.append({"month": r.period_end, "y": y, "case": r.case, "verdict": r.gold_verdict})
    return pd.DataFrame(pts)


STYLE = {
    "standard": {"color": C[0], "lw": 2.2, "label": "стандарт"},
    "incl_one_time": {"color": C[1], "lw": 1.6, "label": "+ внедрение и разовые (D01)"},
    "gross_channel": {"color": C[3], "lw": 1.6, "label": "реселлеры по gross (D05)"},
    "peak_month": {"color": C[2], "lw": 1.4, "ls": "--", "label": "пиковый месяц × 12 (D02)"},
}
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, name in zip(axes, COMPANIES, strict=True):
    s = ARR[name] / 1e6
    x = pd.PeriodIndex(s.index, freq="M").to_timestamp()
    for v in ARR_VARIANTS:
        if v != "standard" and (s[v] - s["standard"]).abs().max() < 1e-3:
            continue  # вариант совпадает со стандартом (нет реселлеров и т. п.)
        ax.plot(x, s[v], **STYLE[v])
    pts = claimed_arr_points(name)
    for k, p in enumerate(pts.itertuples()):
        ax.scatter(
            pd.Period(p.month, "M").to_timestamp(),
            p.y / 1e6,
            s=46,
            zorder=5,
            color=VCOLOR[p.verdict],
            edgecolor="#fcfcfb",
            linewidth=1.5,
            marker="D",
        )
        ax.annotate(
            p.case,
            (pd.Period(p.month, "M").to_timestamp(), p.y / 1e6),
            textcoords="offset points",
            xytext=(-6, 6 - 12 * k),
            ha="right",
            fontsize=7,
            color=INK2,
        )
    ax.set_title(name, loc="left")
    ax.set_ylabel("ARR, млн $" if name == COMPANIES[0] else "")
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
    ax.set_ylim(bottom=0)
handles = [plt.Line2D([], [], **STYLE[s]) for s in ARR_VARIANTS]
handles += [
    plt.Line2D([], [], marker="D", ls="", color=VCOLOR[v], label=f"заявлено: {v}")
    for v in ["SUPPORTED", "EXAGGERATED", "CONTRADICTED"]
]
fig.legend(handles=handles, loc="lower center", ncol=7, bbox_to_anchor=(0.5, -0.08), fontsize=8)
fig.suptitle(
    "ARR по месяцам: стандартное определение и определения компании; ромбы — заявленные значения",
    x=0.01,
    ha="left",
    fontweight="bold",
)
fig.tight_layout()
save(fig, "arr_trajectories")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.0), sharey=True)
for ax, name in zip(axes, COMPANIES, strict=True):
    s = ARR[name]
    x = pd.PeriodIndex(s.index, freq="M").to_timestamp()
    ax.axhspan(0, 5, color="#f0efec", zorder=0)
    ax.axhspan(5, 25, color="#fdf0e9", zorder=0)
    for v in ["incl_one_time", "gross_channel", "peak_month"]:
        g = (s[v] / s["standard"] - 1) * 100
        if g.abs().max() < 0.1:
            continue
        ax.plot(x, g, **STYLE[v])
    ax.axhline(5, color=MUTED, lw=0.8, ls=":")
    ax.axhline(25, color=MUTED, lw=0.8, ls=":")
    ax.set_title(name, loc="left")
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
    ax.set_ylim(-1, 45)
    if name == COMPANIES[0]:
        ax.text(x[0], 1.5, "SUPPORTED ≤ 5%", fontsize=7, color=INK2)
        ax.text(x[0], 20, "EXAGGERATED 5–25%", fontsize=7, color=INK2)
        ax.text(x[0], 27, "CONTRADICTED > 25%", fontsize=7, color=INK2)
axes[0].set_ylabel("ARR варианта / стандарт − 1, %")
fig.legend(
    handles=[
        plt.Line2D([], [], **STYLE[v]) for v in ["incl_one_time", "gross_channel", "peak_month"]
    ],
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.1),
    fontsize=8,
)
fig.suptitle(
    "Насколько определение компании завышает ARR относительно стандарта (зоны допусков P5)",
    x=0.01,
    ha="left",
    fontweight="bold",
)
fig.tight_layout()
save(fig, "definition_gap")

### 3.1. Таблица когорт компании (`cohorts.xlsx`) против леджера и слайды без текста

`cohorts.xlsx` — входной документ data room (P1, FR1): компания сама готовит помесячный MRR по когортам
go-live. Бейзлайн B1 читает эту таблицу как есть, поэтому важно, насколько она отличается от данных.
Сравниваем (а) суммарный MRR таблицы по месяцам с MRR из инвойсов (`DataRoom.customer_mrr`, файлы «как пришли»
минус точные дубли) и (б) NRR за 2025 год, посчитанный по таблице (когорты с go-live ≤ 2024-12: MRR декабря 2025 /
MRR декабря 2024), с NRR библиотеки по стандартному определению.

Отдельно считаем слайды, помеченные `<!-- image-only -->` (график без текстового слоя, FR2), и утверждения на них.

In [ ]:
Y0 = "2024-12"
coh_rows = []
for d in DEV:
    name = d.name
    sheets = load_cohorts(d)
    tab = sheets["cohort_mrr"].set_index("cohort_go_live_month")
    tab.index = tab.index.astype(str)
    tab.columns = tab.columns.astype(str)
    notes = " ".join(sheets["notes"]["note"].astype(str))
    dr = DR[name]
    cm = dr.customer_mrr
    golive = dr.customers.set_index("customer_id")["go_live_date"].dt.strftime("%Y-%m")
    led = cm.groupby(golive.reindex(cm.index)).sum()
    months = [m for m in tab.columns if m in led.columns]
    tot_tab, tot_led = tab[months].sum(), led[months].sum()
    gap = ((tot_tab / tot_led.where(tot_led > 0) - 1) * 100).dropna()
    old = [c for c in tab.index if c <= Y0]
    nrr_tab = tab.loc[old, END].sum() / tab.loc[old, Y0].sum()
    nrr_lib = compute(dr, "nrr", END).value
    slides = split_slides(DOCS[name]["deck.md"])
    img = [s["slide"] for s in slides if s["image_only"]]
    on_slide = np.array(
        [isinstance(loc, dict) and loc.get("slide") in img for loc in CLAIMS["location"]]
    )
    on_img = CLAIMS[(CLAIMS["company_id"] == name) & (CLAIMS["doc"] == "deck.md") & on_slide]
    coh_rows.append(
        {
            "компания": name,
            "определение в notes": "только выжившие" if "retained" in notes else "все клиенты",
            "D06": "D06" in MANIFEST[name]["params"]["cases_planted"],
            "когорт": len(tab),
            "max |разрыв MRR|, %": gap.abs().max(),
            "разрыв MRR 2024-12, %": gap.get(Y0, np.nan),
            "разрыв MRR 2025-12, %": gap.get(END, np.nan),
            "NRR по таблице": nrr_tab,
            "NRR библиотеки": nrr_lib,
            "Δ NRR, п.п.": (nrr_tab - nrr_lib) * 100,
            "слайдов": len(slides),
            "image-only слайдов": len(img),
            "утверждений на image-only": len(on_img),
        }
    )
COH = pd.DataFrame(coh_rows).set_index("компания")
print(COH.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
names = COH.index.tolist()
colors = [C[1] if COH.loc[n, "D06"] else C[0] for n in names]
ax = axes[0]
ax.barh(names, COH["max |разрыв MRR|, %"], color=colors, height=0.6)
for i, v in enumerate(COH["max |разрыв MRR|, %"]):
    ax.text(v, i, f" {v:.1f}%", va="center", fontsize=8, color=INK2)
ax.set_xlabel("max по месяцам |MRR таблицы / MRR инвойсов − 1|, %")
ax.set_title("cohorts.xlsx против инвойсов: MRR", loc="left")
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax = axes[1]
ax.barh(names, COH["Δ NRR, п.п."], color=colors, height=0.6)
for i, v in enumerate(COH["Δ NRR, п.п."]):
    ax.text(v, i, f" {v:+.1f}" if abs(v) >= 0.05 else " 0", va="center", fontsize=8, color=INK2)
for xv in (2, 5):
    ax.axvline(xv, color=MUTED, lw=0.8, ls=":")
ax.set_xlabel("NRR 2025 по таблице компании − NRR библиотеки, п.п. (пунктир — пороги P5: 2 и 5)")
ax.set_title("NRR из таблицы компании", loc="left")
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.legend(
    [plt.Rectangle((0, 0), 1, 1, color=C[1]), plt.Rectangle((0, 0), 1, 1, color=C[0])],
    ["компания с D06 (таблица «только выжившие»)", "остальные"],
    fontsize=7,
    loc="lower right",
)
fig.tight_layout()
save(fig, "cohorts_vs_ledger")

## 4. Качество данных: проверки `dd_verifier.data.quality` и сравнение с реальным прокси

25 именованных проверок на «грязных» файлах dev. Рядом — частоты, измеренные в UCI Online Retail II
(EDA 02, `reports/eda/02_billing_proxy.md`), и целевые значения для B2B SaaS, которые EDA 02 предложил
для генератора (раздел 12 отчёта 02). Числа прокси перенесены из отчёта 02 вручную: сам датасет здесь не читается.

In [ ]:
Q = pd.concat([run_quality_checks(TABLES[n], n) for n in COMPANIES], ignore_index=True)
qa = Q.groupby("check").agg(
    severity=("severity", "first"),
    table=("table", "first"),
    flagged=("n_flagged", "sum"),
    total=("n_total", "sum"),
    companies=("n_flagged", lambda s: int((s > 0).sum())),
)
qa["share_%"] = 100 * qa["flagged"] / qa["total"]
print(qa.sort_values(["flagged"], ascending=False).round(3).to_string())
print("\nошибок уровня error:", int(qa.loc[qa["severity"] == "error", "flagged"].sum()))
per_company = (
    Q.pivot_table(index="check", columns="company_id", values="share", aggfunc="first") * 100
)
print(per_company.loc[qa.index[qa["flagged"] > 0]].round(2).to_string())

# денежный эффект: доля суммы (USD-эквивалент по курсу месяца услуги), а не только строк
money = []
for n in COMPANIES:
    inv = TABLES[n]["invoices"]
    fx = TABLES[n]["fx_rates"]
    inv = inv.assign(fx_month=inv["service_period_start"].dt.strftime("%Y-%m")).merge(
        fx.rename(columns={"month": "fx_month"}), on=["fx_month", "currency"], how="left"
    )
    usd_amt = inv["amount"] * inv["usd_rate"]
    gross = usd_amt[(inv["status"].isin(["paid", "open"])) & (usd_amt > 0)].sum()
    money.append(
        {
            "company": n,
            "credit_notes_%": -usd_amt[inv["status"] == "credit_note"].sum() / gross * 100,
            "void_%": usd_amt[inv["status"] == "void"].sum() / gross * 100,
            "no_customer_id_%": usd_amt[inv["customer_id"].isna() & (usd_amt > 0)].sum()
            / gross
            * 100,
        }
    )
MONEY = pd.DataFrame(money).set_index("company")
print("\nдоля от валовой суммы инвойсов, %:")
print(MONEY.round(2).to_string())

In [ ]:
# Прокси (UCI Online Retail II, отчёт 02) → цель для B2B SaaS (отчёт 02, раздел 12) → синтетика dev
share = qa["share_%"]
CMP = pd.DataFrame(
    [
        ("Точные дубли строк", 1.13, 0.5, share["duplicate_invoice_rows"]),
        ("Пустой customer_id", 16.32, 1.0, share["missing_customer_id"]),
        ("Кредит-ноты / отмены", 15.46, 3.0, share["credit_note_share"]),
        ("Void / технические записи", 0.90, 1.0, share["void_share"]),
        ("Выставлен не в месяц услуги", 0.0, 1.0, share["issued_outside_service_month"]),
    ],
    columns=["проблема", "прокси: Online Retail II", "цель B2B SaaS (EDA 02)", "синтетика dev"],
).set_index("проблема")
print(CMP.round(2).to_string())

fig, ax = plt.subplots(figsize=(8.5, 3.6))
y = np.arange(len(CMP))
h = 0.26
for k, col in enumerate(CMP.columns):
    vals = CMP[col].clip(lower=0.01)
    ax.barh(y + (k - 1) * h, vals, height=h * 0.9, color=C[k], label=col)
    for yi, (v, raw) in enumerate(zip(vals, CMP[col], strict=True)):
        ax.text(v * 1.08, yi + (k - 1) * h, f"{raw:.2g}%", va="center", fontsize=7, color=INK2)
ax.set_xscale("log")
ax.set_yticks(y, CMP.index)
ax.invert_yaxis()
ax.set_xlabel("доля строк / инвойсов, % (лог. шкала)")
ax.grid(axis="y", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
ax.set_title("Шум синтетики против реальной выгрузки инвойсов", loc="left")
fig.tight_layout()
save(fig, "quality_vs_proxy")

### 4.1. Чувствительность метрик и эталона к пустому `customer_id`

EDA 02 предложил 1% инвойсов без `customer_id`. При такой частоте генератор перестал проходить свой
инвариант «шум не меняет эталонный вердикт» (`manifest.json → dirty_vs_clean`). Проверяем, почему:
берём чистые таблицы dev-компаний, 20 раз накладываем шум с разной частотой пустых `customer_id`
(остальные параметры — из `configs/synth.yaml`) и смотрим сдвиг GRR/NRR и долю утверждений,
у которых меняется вердикт. Читаются только конфиги dev-компаний.

In [ ]:
GCFG = load_yaml(CONFIG)
CATALOG = load_catalog()
COMP_CFG = company_configs(GCFG)
ASSIGN = assign_cases(GCFG, COMP_CFG)
DEV_CFG = [c for c in COMP_CFG if c["split"] == "dev"]
CHECKED = {"level", "growth", "share", "count", "customer_mention"}
RATES = [0.0, 0.002, 0.005, 0.01, 0.02, 0.03]
N_SEEDS = 20

t0 = time.time()
sens = []
for ccfg in DEV_CFG:
    res = build_company(ccfg, GCFG, CATALOG, ASSIGN[ccfg["id"]])
    clean = DataRoom(res.clean["customers"], res.clean["invoices"], res.clean["fx_rates"])
    base = {m: compute(clean, m, END).value for m in ["grr", "nrr", "active_customers"]}
    checked = [r for r in res.claims if r["claim_type"] in CHECKED]
    for rate in RATES:
        noise = dict(GCFG["noise"], missing_customer_id_rate=rate)
        for seed in range(N_SEEDS):
            cu, inv, _ = add_noise(
                res.clean["customers"],
                res.clean["invoices"],
                noise,
                END,
                ccfg["invoice_prefix"],
                np.random.default_rng(10_000 + seed),
            )
            dr = DataRoom(cu, drop_exact_duplicates(inv)[0], res.clean["fx_rates"])
            vals = {m: compute(dr, m, END).value for m in base}
            flips = [
                r["claim_id"] for r in checked if evaluate_claim(dr, r).verdict != r["gold_verdict"]
            ]
            sens.append(
                {
                    "company": ccfg["id"],
                    "rate": rate,
                    "seed": seed,
                    "d_grr_pp": (vals["grr"] - base["grr"]) * 100,
                    "d_nrr_pp": (vals["nrr"] - base["nrr"]) * 100,
                    "d_active_%": (vals["active_customers"] / base["active_customers"] - 1) * 100,
                    "flips": len(flips),
                    "n_checked": len(checked),
                    "flipped": ";".join(flips),
                }
            )
SENS = pd.DataFrame(sens)
print(f"{len(SENS)} прогонов за {time.time() - t0:.0f} с")
summary = SENS.groupby("rate").agg(
    grr_median=("d_grr_pp", "median"),
    grr_p05=("d_grr_pp", lambda s: s.quantile(0.05)),
    nrr_median=("d_nrr_pp", "median"),
    nrr_p05=("d_nrr_pp", lambda s: s.quantile(0.05)),
    active_median=("d_active_%", "median"),
    runs_with_flip=("flips", lambda s: (s > 0).mean() * 100),
    flip_rate=("flips", "sum"),
)
summary["flip_rate"] = summary["flip_rate"] / SENS.groupby("rate")["n_checked"].sum() * 100
print(summary.round(2).to_string())
flipped = SENS.loc[SENS["flips"] > 0, "flipped"].str.split(";").explode().value_counts()
print("\nчаще всего меняют вердикт:")
print(flipped.head(8).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
ax = axes[0]
x = np.array(RATES) * 100
for k, (col, label) in enumerate([("d_grr_pp", "GRR"), ("d_nrr_pp", "NRR")]):
    g = SENS.groupby("rate")[col]
    ax.plot(x, g.median(), color=C[k], marker="o", ms=5, label=f"{label}: медиана")
    ax.fill_between(
        x,
        g.quantile(0.05),
        g.quantile(0.95),
        color=C[k],
        alpha=0.15,
        lw=0,
        label=f"{label}: 5–95% прогонов",
    )
ax.axhline(-2, color=MUTED, ls=":", lw=1)
ax.text(
    x[-1], -2.1, "нижний порог P5 для GRR/NRR: 2 п.п.", ha="right", va="top", fontsize=7, color=INK2
)
ax.axvline(GCFG["noise"]["missing_customer_id_rate"] * 100, color=INK2, lw=0.8)
ax.text(
    GCFG["noise"]["missing_customer_id_rate"] * 100 + 0.05,
    0.3,
    "значение в конфиге",
    fontsize=7,
    color=INK2,
)
ax.set_xlabel("доля подписочных инвойсов без customer_id, %")
ax.set_ylabel("сдвиг метрики на 2025-12, п.п.")
ax.set_title("Пустой customer_id занижает удержание", loc="left")
ax.legend(fontsize=7, loc="lower left")

ax = axes[1]
ax.plot(
    x,
    summary["runs_with_flip"],
    color=C[2],
    marker="o",
    ms=5,
    label="прогонов, где сменился ≥ 1 вердикт",
)
ax.plot(
    x, summary["flip_rate"], color=C[1], marker="o", ms=5, label="утверждений со сменой вердикта"
)
ax.set_xlabel("доля подписочных инвойсов без customer_id, %")
ax.set_ylabel("%")
ax.set_ylim(bottom=0)
ax.set_title("Сколько эталонных вердиктов «ломает» шум", loc="left")
ax.legend(fontsize=7, loc="upper left")
fig.tight_layout()
save(fig, "missing_id_sensitivity")

## 5. Утверждения: типы, случаи каталога и баланс эталонных вердиктов

`FACTUAL_OUT_OF_SCOPE` не входит в 5-классовые метрики (вне MVP), но показан отдельно.
Красный флаг = `EXAGGERATED` или `CONTRADICTED`.

In [ ]:
print(pd.crosstab(CLAIMS["company_id"], CLAIMS["gold_verdict"], margins=True).to_string())
print()
print(pd.crosstab(CLAIMS["claim_type"], CLAIMS["gold_verdict"], margins=True).to_string())
print()
print(
    pd.crosstab(
        CLAIMS["doc"].str.replace(r"_0\d", "", regex=True), CLAIMS["gold_verdict"]
    ).to_string()
)
print("\nквалификаторы:", CLAIMS["qualifier"].value_counts().to_dict())
print(
    "семейств:",
    CLAIMS["family_id"].nunique(),
    "| утверждений в семьях из 2+:",
    int(CLAIMS.groupby("family_id")["claim_id"].transform("size").gt(1).sum()),
)

dev_v = CLAIMS["gold_verdict"].value_counts().reindex(VERDICTS, fill_value=0)
hold_v = pd.Series(HOLDOUT["verdict_counts"]).reindex(VERDICTS, fill_value=0)
BAL = pd.DataFrame({"dev": dev_v, "holdout": hold_v})
BAL["dev_%"] = BAL["dev"] / BAL["dev"].sum() * 100
BAL["holdout_%"] = BAL["holdout"] / BAL["holdout"].sum() * 100
print(BAL.round(1).to_string())
for split in ["dev", "holdout"]:
    s = BAL[split].drop("FACTUAL_OUT_OF_SCOPE")
    red = s["EXAGGERATED"] + s["CONTRADICTED"]
    print(
        f"{split}: 5 классов {s.sum()}, красных флагов {red} ({red / s.sum():.1%}), "
        f"SUPPORTED {s['SUPPORTED']} = {s['SUPPORTED'] / s.sum():.1%} от 5 классов"
    )

# Семейство (одно утверждение в нескольких документах) — одно наблюдение
FAM = CLAIMS.groupby("family_id")["gold_verdict"].first()
fam_dev = FAM.value_counts().reindex(VERDICTS, fill_value=0)
fam_hold = pd.Series(HOLDOUT["family_verdict_counts"]).reindex(VERDICTS, fill_value=0)
FAM_BAL = pd.DataFrame({"dev": fam_dev, "holdout": fam_hold})
print("\nсемейств по эталонному вердикту:")
print(FAM_BAL.to_string())
for split in ["dev", "holdout"]:
    s = FAM_BAL[split]
    print(
        f"{split}: семейств-флагов {s['EXAGGERATED'] + s['CONTRADICTED']}, "
        f"семейств с эталоном ≠ SUPPORTED в 5 классах "
        f"{s.drop(['SUPPORTED', 'FACTUAL_OUT_OF_SCOPE']).sum()}"
    )


def wilson(p: float, n: int, z: float = 1.96) -> tuple[float, float]:
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - w, c + w


for label, tab in [("утверждений", BAL), ("семейств", FAM_BAL)]:
    for split in ["dev", "holdout"]:
        n = int(tab.loc[["EXAGGERATED", "CONTRADICTED"], split].sum())
        lo, hi = wilson(0.9, n)
        print(
            f"recall 0,90 на {n} флагах ({label}, {split}): 95% Wilson [{lo:.2f}; {hi:.2f}]"
            " (без учёта кластеризации по компаниям)"
        )
n_fam_hold = int(FAM_BAL.loc[["EXAGGERATED", "CONTRADICTED"], "holdout"].sum())
print(
    f"флагов-семейств на компанию holdout: {n_fam_hold / HOLDOUT['n_companies']:.1f}; "
    f"для ±0,05 (≈140 семейств) нужно ≈{140 / (n_fam_hold / HOLDOUT['n_companies']):.0f} компаний"
)
crit_n = int(FAM_BAL.loc[["EXAGGERATED", "CONTRADICTED"], "holdout"].sum())
print(
    f"critical error 2 из {crit_n} семейств-флагов holdout: верх 95% Wilson {wilson(2 / crit_n, crit_n)[1]:.1%}"
)
for v in ["NOT_ENOUGH_DATA", "UNVERIFIABLE"]:
    print(f"{v}: dev {BAL.loc[v, 'dev']}, holdout {BAL.loc[v, 'holdout']}")

In [ ]:
CASES = [
    "D01",
    "D02",
    "D03",
    "D04",
    "D05",
    "D06",
    "D07",
    "D08",
    "D09",
    "D10",
    "D11",
    "H01",
    "H02",
    "H03",
    "H04",
    "N01",
    "N02",
    "U01",
    "F01",
    "plain",
]
dev_cv = pd.crosstab(CLAIMS["case"], CLAIMS["gold_verdict"]).reindex(
    index=CASES, columns=VERDICTS, fill_value=0
)
hold_cv = pd.DataFrame(HOLDOUT["case_verdicts"]).T.reindex(index=CASES, columns=VERDICTS).fillna(0)
print(pd.concat({"dev": dev_cv, "holdout": hold_cv}, axis=1).astype(int).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4), sharey=True)
for ax, tab, title in [
    (axes[0], dev_cv, f"dev: {len(COMPANIES)} компаний, {len(CLAIMS)} утверждений"),
    (
        axes[1],
        hold_cv,
        f"holdout: {HOLDOUT['n_companies']} компаний, {HOLDOUT['claims_total']} утверждений (агрегаты)",
    ),
]:
    left = np.zeros(len(CASES))
    for v in VERDICTS:
        ax.barh(
            CASES,
            tab[v],
            left=left,
            color=VCOLOR[v],
            height=0.7,
            label=v,
            edgecolor="#fcfcfb",
            linewidth=1,
        )
        left += tab[v].to_numpy()
    for i, total in enumerate(left):
        ax.text(total + 0.5, i, f"{int(total)}", va="center", fontsize=7, color=INK2)
    ax.set_title(title, loc="left")
    ax.set_xlabel("утверждений")
    ax.grid(axis="y", visible=False)
axes[0].invert_yaxis()
handles = [plt.Rectangle((0, 0), 1, 1, color=VCOLOR[v]) for v in VERDICTS]
fig.legend(handles, VERDICTS, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=6, fontsize=8)
fig.suptitle("Эталонные вердикты по случаям каталога", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "claims_by_case")

fig, ax = plt.subplots(figsize=(8, 3.2))
y = np.arange(len(VERDICTS))
for k, (col, label) in enumerate(
    [("dev_%", f"dev ({len(CLAIMS)})"), ("holdout_%", f"holdout ({HOLDOUT['claims_total']})")]
):
    ax.barh(y + (k - 0.5) * 0.38, BAL[col], height=0.34, color=C[k], label=label)
    for yi, (v, n) in enumerate(zip(BAL[col], BAL[col.replace("_%", "")], strict=True)):
        ax.text(
            v + 0.5, yi + (k - 0.5) * 0.38, f"{v:.0f}% ({n})", va="center", fontsize=7, color=INK2
        )
ax.set_yticks(y, VERDICTS)
ax.invert_yaxis()
ax.set_xlabel("% утверждений сплита")
ax.set_xlim(0, 70)
ax.grid(axis="y", visible=False)
ax.legend(fontsize=8, loc="lower right")
ax.set_title("Баланс эталонных вердиктов: dev и holdout", loc="left")
fig.tight_layout()
save(fig, "verdict_balance")

In [ ]:
# Точность значений (гейт G1) против какого эталона мерить: чистые таблицы или файлы «как пришли»
numeric = CLAIMS[
    CLAIMS["claim_type"].isin(["level", "growth", "share", "count"])
    & CLAIMS["true_value_standard"].notna()
].copy()


def value_gap_ok(r) -> bool:
    std, obs = float(r.true_value_standard), float(r.true_value_observed)
    if r.claim_type == "share":
        return abs(obs - std) * 100 <= 0.1  # п.п.
    return abs(obs / std - 1) <= 0.005 if std else obs == std


numeric["within_g1"] = numeric.apply(value_gap_ok, axis=1)
print(
    f"числовых утверждений с рассчитанным значением: {len(numeric)}; значение на файлах «как пришли» "
    f"в пределах допуска G1 от значения на чистых таблицах: {int(numeric['within_g1'].sum())} "
    f"({numeric['within_g1'].mean():.0%})"
)
print(
    numeric.loc[
        ~numeric["within_g1"],
        ["claim_id", "metric_id", "true_value_standard", "true_value_observed"],
    ].to_string()
)
print(
    "noise_sensitive (шум меняет вердикт):",
    int(CLAIMS["noise_sensitive"].sum()),
    "в dev;",
    HOLDOUT["dirty_vs_clean_verdict_mismatches"],
    "в holdout",
)

## 6. Отклонение δ относительно зон допусков P5: сколько утверждений лежат у порогов

Для числовых утверждений берём `delta` из эталона и пороги метрики из `configs/ontology.yaml`.
Чтобы свести метрики с разными порогами (5/25% для уровней, 0,3/1 п.п. для churn, 2/5 п.п. для NRR/GRR и
концентрации) на одну ось, отклонение нормируем кусочно-линейно: 0–1 — зона `SUPPORTED`, 1–2 — между
порогами, > 2 — выше верхнего порога. Знак «+» — отклонение в пользу компании. Если квантификатор
(«more than», «less than») выполнен, отклонение считается нулевым — так работает P5.
Утверждение «у порога», если |δ| отличается от ближайшего порога меньше чем на 25% этого порога.

In [ ]:
num_cl = CLAIMS[
    CLAIMS["claim_type"].isin(["level", "growth", "share", "count"]) & CLAIMS["delta"].notna()
].copy()


def zone_row(r) -> pd.Series:
    as_growth = r.claim_type == "growth" and r.metric_id != "arr_growth"
    tol = tolerance_for(kind="growth") if as_growth else tolerance_for(r.metric_id)
    direction = metric_meta(r.metric_id).get("direction", "up")
    d_unit = float(
        r.delta_p5
    )  # δ по P5: доля для relative, п.п. для pp; 0, если «более/менее N» выполнено
    q_ok = r.qualifier in ("more_than", "less_than") and d_unit == 0.0 and float(r.delta) != 0.0
    fav = (d_unit > 0) if direction == "up" else (d_unit < 0)
    a = abs(d_unit)
    if a <= tol.lower:
        z = a / tol.lower
    elif a <= tol.upper:
        z = 1 + (a - tol.lower) / (tol.upper - tol.lower)
    else:
        z = 2 + min((a - tol.upper) / tol.upper, 1.0)
    near = min(abs(a - tol.lower) / tol.lower, abs(a - tol.upper) / tol.upper) < 0.25
    return pd.Series(
        {
            "z": z if fav else -z,
            "near": near and not q_ok,
            "favourable": fav,
            "qualifier_ok": q_ok,
            "r_lower": a / tol.lower,
            "lower": tol.lower,
            "upper": tol.upper,
            "mode": tol.mode,
        }
    )


num_cl = num_cl.join(num_cl.apply(zone_row, axis=1))
print("числовых утверждений с δ:", len(num_cl))
print("у порога (±25%):", int(num_cl["near"].sum()))
print(
    num_cl.loc[
        num_cl["near"],
        [
            "claim_id",
            "metric_id",
            "case",
            "claimed_value",
            "true_value_standard",
            "delta",
            "lower",
            "upper",
            "gold_verdict",
        ],
    ].to_string()
)
print(
    "\nквантификатор выполнен (delta_p5 = 0, сырое delta ≠ 0):", int(num_cl["qualifier_ok"].sum())
)
r_by_verdict = num_cl.groupby("gold_verdict")["r_lower"].describe()
print("|δ| в долях нижнего порога по вердиктам:")
print(r_by_verdict.round(2).to_string())
print(
    "утверждений с |δ| от 0,6 до 1,4 нижнего порога:",
    int(num_cl["r_lower"].between(0.6, 1.4, inclusive="neither").sum()),
)
red = num_cl[num_cl["gold_verdict"].isin(["EXAGGERATED", "CONTRADICTED"])]
print(
    f"красных флагов среди числовых: {len(red)}, из них отклонение НЕ в пользу компании: "
    f"{int((~red['favourable']).sum())}"
)
print(
    red.loc[
        ~red["favourable"],
        [
            "claim_id",
            "metric_id",
            "case",
            "claimed_value",
            "true_value_standard",
            "delta",
            "gold_verdict",
        ],
    ].to_string()
)
margin = num_cl.loc[num_cl["z"] != 0, "z"].abs()
print("\nрасстояние до ближайшей границы зоны (в нормированных единицах), квартили:")
print(np.minimum((margin - 1).abs(), (margin - 2).abs()).describe().round(2).to_string())

In [ ]:
order = ["SUPPORTED", "EXAGGERATED", "CONTRADICTED"]
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.axvspan(-1, 1, color="#f0efec", zorder=0)
ax.axvspan(1, 2, color="#fdf0e9", zorder=0)
for xv in [-1, 1, 2]:
    ax.axvline(xv, color=MUTED, lw=0.8, ls=":")
rng = np.random.default_rng(0)
for i, v in enumerate(order):
    sub = num_cl[num_cl["gold_verdict"] == v]
    jitter = rng.uniform(-0.18, 0.18, len(sub))
    ax.scatter(
        sub["z"].clip(-3, 3), i + jitter, s=34, color=VCOLOR[v], edgecolor="#fcfcfb", lw=1, zorder=3
    )
    nr = sub[sub["near"]]
    ax.scatter(
        nr["z"].clip(-3, 3),
        i + jitter[sub["near"].to_numpy()],
        s=110,
        facecolor="none",
        edgecolor=INK,
        lw=1,
        zorder=4,
    )
ax.set_yticks(range(3), order)
ax.set_ylim(2.5, -0.85)
ax.set_xlim(-3.2, 3.2)
ax.set_xticks(
    [-3, -2, -1, 0, 1, 2, 3], ["≤ −2·верх.", "−верх.", "−ниж.", "0", "ниж.", "верх.", "≥ 2·верх."]
)
ax.set_xlabel("нормированное отклонение δ: ← не в пользу компании | в пользу компании →")
ax.grid(axis="y", visible=False)
ax.text(0, -0.65, "SUPPORTED", ha="center", fontsize=7, color=INK2)
ax.text(1.5, -0.65, "EXAGGERATED", ha="center", fontsize=7, color=INK2)
ax.text(2.6, -0.65, "CONTRADICTED", ha="center", fontsize=7, color=INK2)
ax.text(-2.1, -0.65, "CONTRADICTED", ha="center", fontsize=7, color=INK2)
ax.set_title(
    f"Числовые утверждения dev ({len(num_cl)}) в зонах P5; обведены — у порога (±25%)", loc="left"
)
fig.tight_layout()
save(fig, "delta_zones")

## 7. Матрица «метрика × нужные таблицы и колонки × наличие в dev»

Источник требований — поле `required_data` онтологии. Наличие: колонка есть во всех пяти dev-компаниях;
полнота — доля непустых значений там, где пустое значение означает дефект (у `go_live_date` и
`churn_date` пустота — норма: не запущен / не ушёл). История: хватает ли месяцев до 2025-12 (NRR/GRR —
13 месяцев, `peak_month` и концентрация — 12). Метрики вне онтологии требуют данных, которых в data room
нет (затраты, COGS, денежный поток) → `NOT_ENOUGH_DATA`.

In [ ]:
ONT = yaml.safe_load(Path("configs/ontology.yaml").read_text(encoding="utf-8"))
SEMANTIC_NULL = {("customers", "go_live_date"), ("customers", "churn_date")}
HISTORY = {"nrr": 13, "grr": 13, "top10_concentration": 12, "logo_churn": 13, "arr_growth": 13}


def completeness(table: str, col: str) -> float:
    vals = []
    for n in COMPANIES:
        df = TABLES[n][table]
        if col not in df.columns:
            return 0.0
        if (table, col) in SEMANTIC_NULL:
            vals.append(1.0)
        else:
            vals.append(float(df[col].notna().mean()))
    return min(vals)


req_rows = []
for m in ONT["metrics"]:
    for table, cols in m["required_data"].items():
        for col in cols:
            req_rows.append(
                {"metric": m["id"], "element": f"{table}.{col}", "value": completeness(table, col)}
            )
    need = HISTORY.get(m["id"], 1)
    hist = min(len(month_range(*DR[n].coverage)) for n in COMPANIES)
    req_rows.append(
        {
            "metric": m["id"],
            "element": f"история ≥ {need} мес.",
            "value": 1.0 if hist >= need else 0.5,
        }
    )
for m in ONT["out_of_ontology"]:
    req_rows.append({"metric": m["id"], "element": m["missing_data"], "value": 0.0})
REQ = pd.DataFrame(req_rows)
REQ["element"] = REQ["element"].str.replace(r"история ≥ \d+ мес\.", "история на дату", regex=True)
MAT = REQ.pivot_table(index="metric", columns="element", values="value", aggfunc="min")
metric_order = [m["id"] for m in ONT["metrics"]] + [m["id"] for m in ONT["out_of_ontology"]]
col_order = list(REQ["element"].drop_duplicates())
MAT = MAT.reindex(index=metric_order, columns=col_order)
claims_per_metric = CLAIMS.groupby("metric_id").size()
n02 = CLAIMS[CLAIMS["case"] == "N02"][["claim_id", "metric_id", "period_end", "quote"]]
print(MAT.round(4).to_string())
print("\nутверждений dev по метрикам:", claims_per_metric.to_dict())
print("\nN02 (метрика в онтологии, но нет истории на дату):")
print(n02.to_string())
print("\nпокрытие историей:", {n: DR[n].coverage for n in COMPANIES})

In [ ]:
GOOD, WARN, CRIT = "#0ca30c", "#fab219", "#d03b3b"
fig, ax = plt.subplots(figsize=(13, 5.6))
nrow, ncol = MAT.shape
for i, metric in enumerate(MAT.index):
    for j, el in enumerate(MAT.columns):
        v = MAT.loc[metric, el]
        if pd.isna(v):
            continue
        color = GOOD if v >= 0.999 else (WARN if v > 0 else CRIT)
        ax.add_patch(plt.Rectangle((j + 0.06, i + 0.06), 0.88, 0.88, color=color, lw=0))
        label = "✓" if v >= 0.999 else (f"{v * 100:.1f}" if v > 0 else "нет")
        ax.text(
            j + 0.5,
            i + 0.5,
            label,
            ha="center",
            va="center",
            fontsize=7,
            color="#0b0b0b" if color == WARN else "white",
        )
ax.set_xlim(0, ncol)
ax.set_ylim(nrow, 0)
ax.set_xticks(np.arange(ncol) + 0.5, MAT.columns, rotation=55, ha="right", fontsize=7)
labels = [f"{m} ({claims_per_metric.get(m, 0)})" for m in MAT.index]
ax.set_yticks(np.arange(nrow) + 0.5, labels, fontsize=8)
ax.grid(False)
ax.axhline(len(ONT["metrics"]), color=INK2, lw=1)
for s in ["left", "bottom"]:
    ax.spines[s].set_visible(False)
ax.tick_params(length=0)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in [GOOD, WARN, CRIT]]
ax.legend(
    handles,
    ["нужно и есть (100%)", "нужно, есть частично (% непустых)", "нужно, в data room нет"],
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=3,
    fontsize=8,
)
ax.set_title(
    "Какие данные нужны для какой метрики и есть ли они в dev (в скобках — утверждений dev)",
    loc="left",
    pad=26,
)
fig.tight_layout()
save(fig, "metric_data_matrix")

## 8. Формулировки: синтетика против реальных проспектов SEC

**Идеальный вариант** — сравнить с корпусом предложений `data/reference/sec_kpi_claims.csv` (EDA 01):
доли метрик, квалификаторов и числовых форматов. Корпус собирается скриптом `scripts/fetch_sec_claims.py`,
которому для скачивания документов с www.sec.gov нужен `SEC_USER_AGENT` с контактом. Если файла нет, ячейка ниже
это сообщает и сравнение по предложениям пропускается.

**Что есть уже сейчас:**

1. тот же rule-based экстрактор из `scripts/fetch_sec_claims.py`, применённый к синтетическим предложениям, —
   признаки в одних координатах с будущим корпусом SEC (семейство метрики, единица, квалификатор, тип периода);
2. частоты терминов в проспектах S-1/424B4 за 2019–2025 по полнотекстовому поиску EDGAR, все отрасли и
   SIC 737x (ПО). Их собирает EDA 01 в `data/reference/sec_efts_term_counts.csv`; здесь файл только читается.

In [ ]:
spec = importlib.util.spec_from_file_location(
    "fetch_sec_claims", Path("scripts/fetch_sec_claims.py")
)
fsc = importlib.util.module_from_spec(spec)
sys.modules["fetch_sec_claims"] = fsc
spec.loader.exec_module(fsc)

FAMILY_OF = {
    "arr": "arr",
    "arr_growth": "arr",
    "mrr": "mrr",
    "nrr": "nrr",
    "grr": "grr",
    "logo_churn": "churn",
    "active_customers": "customers",
    "revenue": "revenue",
    "acv": "other",
    "top10_concentration": "other",
}


def number_format(text: str) -> str:
    if re.search(r"\$\s?\d[\d,.]*\s?(million|billion)", text, re.I):
        return "$X million/billion"
    if re.search(r"\$\s?\d[\d,.]*\s?[KMB]\b", text):
        return "$XK / $XM"
    if re.search(r"\$\s?\d", text):
        return "$X (полностью)"
    if re.search(r"\d\s?(%|percent)", text, re.I):
        return "X%"
    if re.search(r"\d(\.\d+)?\s?(x|times)\b", text, re.I):
        return "Nx"
    if re.search(r"\d\+", text):
        return "N+"
    if re.search(r"\d", text):
        return "число"
    return "без числа"


syn = CLAIMS[CLAIMS["claim_type"].isin(["level", "growth", "share", "count"])].copy()
ext = syn["quote"].apply(fsc.extract_claim)
syn["sec_family"] = ext.apply(lambda c: c.metric_family if c else None)
syn["sec_qualifier"] = ext.apply(lambda c: c.qualifier if c else None)
syn["sec_period"] = ext.apply(lambda c: c.period_type if c else None)
syn["gold_family"] = syn["metric_id"].map(FAMILY_OF)
syn["format"] = syn["quote"].apply(number_format)
hit = syn["sec_family"].notna()
same = syn["sec_family"] == syn["gold_family"]
print(
    f"синтетических числовых утверждений: {len(syn)}; экстрактор SEC нашёл KPI в {hit.mean():.0%}, "
    f"семейство совпало с эталоном в {same.mean():.0%}"
)
print(pd.crosstab(syn["gold_family"], syn["sec_family"].fillna("— не найдено")).to_string())
print("\nквалификатор по экстрактору:", syn["sec_qualifier"].value_counts(dropna=False).to_dict())
print("тип периода по экстрактору:", syn["sec_period"].value_counts(dropna=False).to_dict())
print("числовые форматы:", syn["format"].value_counts().to_dict())
print(
    "доля утверждений с явным периодом в тексте:",
    f"{syn['quote'].str.contains(r'20\d\d|year over year|months', regex=True).mean():.0%}",
)
nrr_names = (
    syn.loc[syn["metric_id"] == "nrr", "quote"]
    .str.extract(r"(?i)(net revenue retention|NRR|net dollar retention|dollar-based net)")[0]
    .str.lower()
)
print("как названа NRR в синтетике:", nrr_names.value_counts().to_dict())

In [ ]:
SEC_CSV = Path("data/reference/sec_kpi_claims.csv")
if SEC_CSV.exists():
    sec = pd.read_csv(SEC_CSV)
    sec["format"] = sec["text"].apply(number_format)
    print("SEC: предложений", len(sec), "компаний", sec["company"].nunique())
    print(
        pd.concat(
            {
                "SEC": sec["metric_family"].value_counts(normalize=True),
                "синтетика": syn["sec_family"].value_counts(normalize=True),
            },
            axis=1,
        )
        .round(3)
        .to_string()
    )
    print(
        pd.concat(
            {
                "SEC": sec["qualifier"].value_counts(normalize=True),
                "синтетика": syn["sec_qualifier"].value_counts(normalize=True),
            },
            axis=1,
        )
        .round(3)
        .to_string()
    )
    print(
        pd.concat(
            {
                "SEC": sec["format"].value_counts(normalize=True),
                "синтетика": syn["format"].value_counts(normalize=True),
            },
            axis=1,
        )
        .round(3)
        .to_string()
    )
    print("доля определений в SEC:", round(sec["is_definition"].mean(), 3))
else:
    sec = None
    print(
        f"{SEC_CSV} нет: корпус SEC ещё не собран (нужен SEC_USER_AGENT с контактом, см. EDA 01)."
    )
    print(
        "Сравнение по предложениям пропущено; ниже — частоты терминов по полнотекстовому поиску EDGAR."
    )

In [ ]:
TERMS_CSV = Path("data/reference/sec_efts_term_counts.csv")
TERM_COUNTS = pd.read_csv(TERMS_CSV) if TERMS_CSV.exists() else None
if TERM_COUNTS is None:
    print(f"{TERMS_CSV} нет: сначала выполните notebooks/01_eda_sec_claims.ipynb")
else:
    print(
        TERM_COUNTS[["group", "term", "n_documents", "n_documents_sic737x"]].to_string(index=False)
    )
    print(
        TERM_COUNTS.groupby("group")[["n_documents", "n_documents_sic737x"]]
        .sum()
        .sort_values("n_documents_sic737x", ascending=False)
        .to_string()
    )

In [ ]:
# База правой панели — все утверждения dev с metric_id (как в матрице раздела 7) и вне онтологии
with_metric = CLAIMS[CLAIMS["metric_id"].notna()]
syn_terms = {
    "Выручка": with_metric["metric_id"].eq("revenue").sum(),
    "Клиенты": with_metric["metric_id"].eq("active_customers").sum(),
    "NRR": with_metric["metric_id"].eq("nrr").sum(),
    "GRR": with_metric["metric_id"].eq("grr").sum(),
    "Churn": with_metric["metric_id"].eq("logo_churn").sum(),
    "ACV": with_metric["metric_id"].eq("acv").sum(),
    "Концентрация": with_metric["metric_id"].eq("top10_concentration").sum(),
    "ARR / MRR": with_metric["metric_id"].isin(["arr", "mrr", "arr_growth"]).sum(),
    "вне онтологии": CLAIMS["claim_type"].eq("out_of_ontology").sum(),
}
print("утверждений dev по группам:", {k: int(v) for k, v in syn_terms.items()})
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.4, 1]})
ax = axes[0]
if TERM_COUNTS is not None:
    tc = TERM_COUNTS.sort_values(["group", "n_documents_sic737x"], ascending=[True, True])
    groups = list(dict.fromkeys(tc["group"]))
    gcolor = {g: C[i % len(C)] for i, g in enumerate(groups)}
    ax.barh(
        tc["term"], tc["n_documents_sic737x"], color=[gcolor[g] for g in tc["group"]], height=0.7
    )
    for i, v in enumerate(tc["n_documents_sic737x"]):
        ax.text(v, i, f" {v}", va="center", fontsize=7, color=INK2)
    ax.set_xlabel("документов S-1 / 424B4 с SIC 737x за 2019–2025 с точной фразой")
    ax.legend(
        [plt.Rectangle((0, 0), 1, 1, color=gcolor[g]) for g in groups],
        groups,
        fontsize=7,
        loc="center right",
    )
    ax.set_xlim(0, tc["n_documents_sic737x"].max() * 1.4)
    ax.grid(axis="y", visible=False)
    ax.set_title("Проспекты ПО (SIC 737x): как называют SaaS-метрики", loc="left")
ax = axes[1]
s = pd.Series(syn_terms).sort_values()
ax.barh(s.index, s.values, color=C[0], height=0.6)
for i, v in enumerate(s.values):
    ax.text(v, i, f" {v}", va="center", fontsize=7, color=INK2)
ax.set_xlabel(f"утверждений dev (все {len(with_metric)} с метрикой онтологии + вне онтологии)")
ax.grid(axis="y", visible=False)
ax.set_title("Синтетика dev: утверждения по метрикам", loc="left")
fig.tight_layout()
save(fig, "sec_vs_synth_terms")

## Итог

Выводы с числами и ссылками на графики — в `reports/eda/03_synthetic_dataroom.md`.